# Chapter 6 Lab — Reimplement TinySleepNet

**Think of it like this:** you've already met both halves of this separately. Workshop 1 ("Robo Learns to Spot a Spindle") was the CNN half — a stamp that slides over ONE epoch's raw signal and summarizes it into a few numbers. Workshop 2 ("Robo Remembers") was the LSTM half — a journal that reads a SEQUENCE of those summaries, one epoch at a time, and carries memory forward. TinySleepNet is just those two things, bolted together, trained as one model: CNN first, epoch by epoch, then LSTM on top, reading the CNN's summaries in order.

This is a **simplified** version — smaller and faster than the real TinySleepNet paper, built to actually finish training in a few minutes on Colab's free tier, not to match the paper's published numbers exactly. That's a deliberate, honest trade: realistic for a laptop lab, not a faithful line-by-line reproduction.

**Goal of this lab:** train a real CNN+LSTM on raw EEG, then compare it directly against Chapter 4's Random Forest baseline on the exact same Alice/Bob subject split.

In [ ]:
!pip install -q mne torch scikit-learn
%matplotlib inline
import mne
import numpy as np
import torch
import torch.nn as nn
from mne.datasets.sleep_physionet.age import fetch_data

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## Same Data, Same Split as Chapter 4

Alice trains, Bob tests. Never mixed — same rule, every chapter.

In [ ]:
ALICE, BOB = 0, 1
[alice_files, bob_files] = fetch_data(subjects=[ALICE, BOB], recording=[1])

annotation_desc_2_event_id = {
    "Sleep stage W": 1, "Sleep stage 1": 2, "Sleep stage 2": 3,
    "Sleep stage 3": 4, "Sleep stage 4": 4, "Sleep stage R": 5,
}
event_id = {"Wake": 1, "N1": 2, "N2": 3, "N3": 4, "REM": 5}
id_to_name = {v: k for k, v in event_id.items()}
N_CLASSES = 5

def load_raw_epochs(files):
    raw = mne.io.read_raw_edf(files[0], stim_channel="Event marker", infer_types=True, preload=True, verbose="error")
    raw.pick(raw.ch_names[0:1])  # single EEG channel, keeps this laptop-realistic
    annot = mne.read_annotations(files[1])
    raw.set_annotations(annot, emit_warning=False)
    annot.crop(annot[1]["onset"] - 30 * 60, annot[-2]["onset"] + 30 * 60)
    raw.set_annotations(annot, emit_warning=False)
    events, _ = mne.events_from_annotations(raw, event_id=annotation_desc_2_event_id, chunk_duration=30.0)
    tmax = 30.0 - 1.0 / raw.info["sfreq"]
    epochs = mne.Epochs(raw=raw, events=events, event_id=event_id, tmin=0.0, tmax=tmax, baseline=None, preload=True)
    X = epochs.get_data()[:, 0, :]          # (n_epochs, n_times) -- raw signal, NOT band power
    y = epochs.events[:, 2] - 1             # zero-index the 5 classes
    X = (X - X.mean()) / (X.std() + 1e-8)   # simple global normalization
    return X.astype(np.float32), y.astype(np.int64)

X_train, y_train = load_raw_epochs(alice_files)
X_test, y_test = load_raw_epochs(bob_files)
print("Train (Alice):", X_train.shape, "  Test (Bob):", X_test.shape)

## Chunk Into Sequences

The LSTM half needs a *sequence* of epochs, not one at a time — that's the whole point of Workshop 2's journal. We chunk each night into back-to-back windows of 10 epochs (5 minutes each). The real TinySleepNet paper uses longer sequences (~20–25 epochs); 10 keeps training fast enough to actually finish in this lab.

In [ ]:
SEQ_LEN = 10

def make_sequences(X, y, seq_len=SEQ_LEN):
    n = len(X) // seq_len
    X = X[: n * seq_len].reshape(n, seq_len, -1)
    y = y[: n * seq_len].reshape(n, seq_len)
    return torch.from_numpy(X), torch.from_numpy(y)

X_train_seq, y_train_seq = make_sequences(X_train, y_train)
X_test_seq, y_test_seq = make_sequences(X_test, y_test)
print("Train sequences:", X_train_seq.shape, "  Test sequences:", X_test_seq.shape)

## The Model: Workshop 1's Stamp, Then Workshop 2's Journal

`cnn` runs on every epoch separately — exactly like dragging Robo's stamp across one signal at a time — and squeezes each 30-second epoch down to a short list of numbers (a *feature embedding*). `lstm` then reads that whole sequence of summaries in order, the same way Robo's journal read one epoch's worth of history at a time. `classifier` turns the journal's output, at each step, into a guess among the 5 stages.

In [ ]:
class TinySleepNetLite(nn.Module):
    def __init__(self, n_classes=N_CLASSES, cnn_channels=16, lstm_hidden=32):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1, cnn_channels, kernel_size=50, stride=6, padding=22),
            nn.BatchNorm1d(cnn_channels),
            nn.ReLU(),
            nn.MaxPool1d(8, 8),
            nn.Conv1d(cnn_channels, cnn_channels * 2, kernel_size=8, stride=1, padding=4),
            nn.BatchNorm1d(cnn_channels * 2),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),  # Workshop 1's "zoom out" (pooling), collapsed to one summary number per filter
        )
        self.lstm = nn.LSTM(input_size=cnn_channels * 2, hidden_size=lstm_hidden, batch_first=True)
        self.classifier = nn.Linear(lstm_hidden, n_classes)

    def forward(self, x):
        # x: (batch, seq_len, n_times)
        batch, seq_len, n_times = x.shape
        x = x.view(batch * seq_len, 1, n_times)       # treat every epoch independently for the CNN stage
        feats = self.cnn(x).squeeze(-1)                # (batch*seq_len, cnn_channels*2)
        feats = feats.view(batch, seq_len, -1)          # back into sequences for the LSTM
        lstm_out, _ = self.lstm(feats)                  # (batch, seq_len, lstm_hidden)
        return self.classifier(lstm_out)                # (batch, seq_len, n_classes)

model = TinySleepNetLite().to(device)
print(model)

## Train It

A handful of passes over Alice's night is enough to see real learning happen — watch the loss go down.

In [ ]:
X_train_seq, y_train_seq = X_train_seq.to(device), y_train_seq.to(device)
X_test_seq, y_test_seq = X_test_seq.to(device), y_test_seq.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

N_ROUNDS = 15
for round_num in range(N_ROUNDS):
    model.train()
    optimizer.zero_grad()
    logits = model(X_train_seq)                                   # (batch, seq_len, n_classes)
    loss = loss_fn(logits.reshape(-1, N_CLASSES), y_train_seq.reshape(-1))
    loss.backward()
    optimizer.step()
    if round_num % 3 == 0 or round_num == N_ROUNDS - 1:
        print(f"Round {round_num+1}/{N_ROUNDS}  loss={loss.item():.3f}")

## Compare Against Chapter 4's Baseline

Same subject split, same test set (Bob). This is the real comparison the whole chapter is building toward.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

model.eval()
with torch.no_grad():
    test_logits = model(X_test_seq)
    test_pred = test_logits.argmax(dim=-1).cpu().numpy().reshape(-1)
test_true = y_test_seq.cpu().numpy().reshape(-1)

accuracy = (test_pred == test_true).mean()
print(f"TinySleepNetLite accuracy on Bob: {accuracy:.1%}")
print("\nCompare this to Chapter 4's Random Forest accuracy on the exact same Bob test set.")

labels = sorted(id_to_name.keys())
names = [id_to_name[l] for l in labels]
cm = confusion_matrix(test_true + 1, test_pred + 1, labels=labels)  # +1 to match event_id's 1-indexing
print("\nConfusion matrix (rows = true, cols = predicted):")
print("       " + "  ".join(f"{n:>5}" for n in names))
for i, row in enumerate(cm):
    print(f"{names[i]:>6} " + "  ".join(f"{v:>5}" for v in row))

print()
print(classification_report(test_true + 1, test_pred + 1, labels=labels, target_names=names, zero_division=0))

## Done when...

You have a trained CNN+LSTM model and a clear comparison against Chapter 4's baseline: overall accuracy, per-stage scores, and specifically whether N1/REM confusion got better, worse, or stayed about the same. With only 15 training rounds and one subject's worth of data, don't expect to beat the baseline by a huge margin yet — the real TinySleepNet is trained far longer, on far more subjects. The point of this lab is watching the two workshops' ideas actually work together on real data, not chasing a leaderboard number.

Chapter 7 reads the architecture this one is a simplified cousin of — DeepSleepNet — critically, alongside one more design. Chapter 8 connects all of this back to the real TinySleepNet code already sitting inside Dreamento's own repository.